🔹 Step 1: Tải lại file macro từ Kaggle (nếu chưa có)

In [ ]:
# Imports
import os
import pandas as pd
import kagglehub
from google.cloud import storage

# 🔐 GCP key
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/gcp-key.json"
print("✅ GCP credentials set.")

# 📥 Tải macro dataset từ KaggleHub
path = kagglehub.dataset_download("macrosynergy/fixed-income-returns-and-macro-trends")
macro_csv_path = os.path.join(path, "JPMaQS_Quantamental_Indicators.csv")

# 📊 Load dữ liệu và in cột
df = pd.read_csv(macro_csv_path)
print("✅ File loaded. Dữ liệu có dạng:")
print("📌 Columns:", df.columns.tolist())
print("📌 Số dòng:", len(df))
print("📌 5 dòng đầu:")
df.head()


✅ GCP credentials set.


100%|██████████| 31.4M/31.4M [00:01<00:00, 22.1MB/s]

Extracting files...


✅ File loaded. Dữ liệu có dạng:
📌 Columns: ['Unnamed: 0', 'real_date', 'cid', 'xcat', 'value', 'grading', 'eop_lag', 'mop_lag']
📌 Số dòng: 3390059
📌 5 dòng đầu:


,Unnamed: 0,real_date,cid,xcat,value,grading,eop_lag,mop_lag
0,0,2000-01-03,AUD,CPIC_SA_P1M1ML12,1.244168,2.0,95.0,292.0
1,1,2000-01-04,AUD,CPIC_SA_P1M1ML12,1.244168,2.0,96.0,293.0
2,2,2000-01-05,AUD,CPIC_SA_P1M1ML12,1.244168,2.0,97.0,294.0
3,3,2000-01-06,AUD,CPIC_SA_P1M1ML12,1.244168,2.0,98.0,295.0
4,4,2000-01-07,AUD,CPIC_SA_P1M1ML12,1.244168,2.0,99.0,296.0


🔹 Step 2: Lọc macro dữ liệu cho UK và chỉ số cần dùng

In [ ]:
df_uk = df[df['cid'] == 'GBP'].copy()
df_uk['date'] = pd.to_datetime(df_uk['real_date'])
df_uk['Year'] = df_uk['date'].dt.year
df_uk['Month'] = df_uk['date'].dt.month

selected_xcats = [
    "CPIH_SA_P1M1ML12",          # ✅ Inflation
    "RGDP_SA_P1Q1QL4_20QMA",     # 🟡 Smoothed GDP Growth
    "RYLDIRS05Y_NSA"             # 🟡 5Y Interest Rate Yield (thay thế POLICYRATE)
]
df_selected = df_uk[df_uk["xcat"].isin(selected_xcats)].copy()
df_uk = df[df['cid'] == 'GBP']
print("🧾 Available xcat (UK):")
print(df_uk["xcat"].value_counts().head(30))

🧾 Available xcat (UK):
xcat
CPIC_SA_P1M1ML12                6248
CPIC_SJA_P3M3ML3AR              6248
CPIC_SJA_P6M6ML6AR              6248
CPIH_SA_P1M1ML12                6248
CPIH_SJA_P3M3ML3AR              6248
CPIH_SJA_P6M6ML6AR              6248
DU02YXR_NSA                     6248
DU02YXR_VT10                    6248
DU05YXR_NSA                     6248
DU05YXR_VT10                    6248
EQXR_NSA                        6248
EQXR_VT10                       6248
FXTARGETED_NSA                  6248
FXXR_NSA                        6248
FXUNTRADABLE_NSA                6248
FXXR_VT10                       6248
INFTEFF_NSA                     6248
RYLDIRS05Y_NSA                  6248
INTRGDP_NSA_P1M1ML12_3MMA       6248
INTRGDPv5Y_NSA_P1M1ML12_3MMA    6248
RGDP_SA_P1Q1QL4_20QMA           6248
RYLDIRS02Y_NSA                  6248
PCREDITBN_SJA_P1M1ML12          5903
PCREDITGDP_SJA_D1M1ML12         5903
FXCRR_NSA                       5712
Name: count, dtype: int64


🔹 Step 3: Pivot thành bảng có mỗi macro là 1 cột

In [ ]:
pivot_macro = df_selected.pivot_table(
    index=["Year", "Month"],
    columns="xcat",
    values="value"
).reset_index()

pivot_macro.to_csv("/content/uk_macro_data.csv", index=False)
print("✅ Saved pivoted macro to /content/uk_macro_data.csv")


✅ Saved pivoted macro to /content/uk_macro_data.csv


🔹 Step 4: Upload file lên GCS

In [ ]:
from google.cloud import storage
import os

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/gcp-key.json"
bucket_name = "boothill2001-dataset"
client = storage.Client()
bucket = client.bucket(bucket_name)

macro_blob = bucket.blob("uk_property_data/processed/uk_macro_data.csv")
macro_blob.upload_from_filename("/content/uk_macro_data.csv")
print("✅ Uploaded macro file to GCS")


✅ Uploaded macro file to GCS


🔹 Step 5: Merge lại với file bất động sản

In [ ]:
real_path = "/content/cleaned_real_estate_full.csv"
real_blob = bucket.blob("uk_property_data/processed/cleaned_real_estate_full.csv")

if not os.path.exists(real_path):
    real_blob.download_to_filename(real_path)
    print("✅ Downloaded cleaned real estate file")

# Merge
df_real = pd.read_csv(real_path)
df_macro = pd.read_csv("/content/uk_macro_data.csv")
df_merged = df_real.merge(df_macro, on=["Year", "Month"], how="left")
df_merged.to_csv("/content/merged_real_estate_macro.csv", index=False)
print("✅ Merged shape:", df_merged.shape)


✅ Merged shape: (28276227, 30)


🔹 Step 6: Upload file merged mới lên GCS + lưu danh sách cột

In [ ]:
# Upload merged
merged_blob = bucket.blob("uk_property_data/processed/merged_real_estate_macro.csv")
merged_blob.upload_from_filename("/content/merged_real_estate_macro.csv")

# Save column names
import json
json.dump(list(df_merged.columns), open("/content/feature_names_full.json", "w"))

# Upload JSON
feat_blob = bucket.blob("uk_property_data/models/feature_names_full.json")
feat_blob.upload_from_filename("/content/feature_names_full.json")

print("✅ Uploaded merged dataset + feature_names_full.json to GCS")

✅ Uploaded merged dataset + feature_names_full.json to GCS


In [ ]:
print("📌 Final columns:", df_merged.columns.tolist())

📌 Final columns: ['Transaction_ID', 'Price', 'Date_of_Transfer', 'Postcode', 'Property_Type', 'Old/New', 'Duration', 'PAON', 'Street', 'Locality', 'Town/City', 'District', 'County', 'PPDCategory_Type', 'Record_Status', 'Year', 'Month', 'Quarter', 'Region_Code', 'Season', 'Is_Weekend', 'High_Value_Property', 'Holiday_Season', 'Urban_Rural', 'Log_Price', 'Price_Change_Ratio', 'Anomaly', 'CPIH_SA_P1M1ML12', 'RGDP_SA_P1Q1QL4_20QMA', 'RYLDIRS05Y_NSA']


In [ ]:
import pandas as pd
import os
from google.cloud import storage

# === GCP CONFIG ===
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/gcp-key.json"
bucket_name = "boothill2001-dataset"
csv_gcs_path = "uk_property_data/processed/merged_real_estate_macro.csv"
parquet_gcs_path = "uk_property_data/processed/merged_real_estate_macro.parquet"

csv_local_path = "/content/merged_real_estate_macro.csv"
parquet_local_path = "/content/merged_real_estate_macro.parquet"

# === Step 1: Download CSV from GCS if not exists ===
def download_csv():
    if os.path.exists(csv_local_path):
        print(f"✅ Found local file: {csv_local_path}")
    else:
        print("⬇️ Downloading CSV from GCS...")
        client = storage.Client()
        bucket = client.bucket(bucket_name)
        blob = bucket.blob(csv_gcs_path)
        blob.download_to_filename(csv_local_path)
        print(f"✅ Downloaded: {csv_gcs_path} → {csv_local_path}")

# === Step 2: Convert to Parquet ===
def convert_to_parquet():
    df = pd.read_csv(csv_local_path)
    df.to_parquet(parquet_local_path, index=False)
    print(f"✅ Converted to: {parquet_local_path}")

# === Step 3: Upload Parquet to GCS ===
def upload_parquet():
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(parquet_gcs_path)
    blob.upload_from_filename(parquet_local_path)
    print(f"✅ Uploaded to GCS: gs://{bucket_name}/{parquet_gcs_path}")

# 🔁 Run full pipeline
download_csv()
convert_to_parquet()
upload_parquet()


⬇️ Downloading CSV from GCS...
✅ Downloaded: uk_property_data/processed/merged_real_estate_macro.csv → /content/merged_real_estate_macro.csv
✅ Converted to: /content/merged_real_estate_macro.parquet
✅ Uploaded to GCS: gs://boothill2001-dataset/uk_property_data/processed/merged_real_estate_macro.parquet
